In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch-scatter torch-sparse torch-geometric \
    -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install pytorch-crf seqeval


In [ ]:
import os
import json
import csv
import torch
import torch.nn as nn
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F

from torchcrf import CRF
from torch_geometric.nn import GATv2Conv
from torch.utils.data import Dataset, DataLoader
from torch.nn import TransformerDecoderLayer, TransformerDecoder
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (
    classification_report, f1_score,
    precision_recall_fscore_support
)


In [ ]:
PHOBERT       = "vinai/phobert-large"
MAX_LEN       = 128
BATCH         = 32
EPOCHS        = 15
LR            = 5e-5
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
WARMUP_RATIO  = 0.1
DROP_PROB     = 0.3
GAT_HEADS     = 8
HIDDEN_DIM    = 256
WINDOW_SIZE   = 4
OUTPUT_DIR    = "/content/drive/MyDrive/ViMedNer/output/gat_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
def merge_entity_label(label: str) -> str:
    if label.startswith("B-") or label.startswith("I-"):
        return label.split("-", 1)[1]
    return label


def load_conll_data(file_path: str):
    sentences, labels = [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        words, tags = [], []
        for line in f:
            line = line.strip()
            if not line:
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
                continue
            splits = line.split()
            if len(splits) >= 2:
                words.append(splits[0])
                tags.append(splits[-1])
        if words:
            sentences.append(words)
            labels.append(tags)
    return sentences, labels


In [ ]:
class TextGraphDataset(Dataset):

    def __init__(self, sentences, labels, tokenizer, label2id, max_len):
        self.sentences = sentences
        self.labels    = labels
        self.tokenizer = tokenizer
        self.label2id  = label2id
        self.max_len   = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words       = self.sentences[idx]
        word_labels = self.labels[idx]

        tokens, label_ids = [], []

        for i, word in enumerate(words):
            word_tokens = self.tokenizer.tokenize(word)
            if not word_tokens:
                word_tokens = [self.tokenizer.unk_token]

            for j, tok in enumerate(word_tokens):
                tokens.append(tok)
                if j == 0:
                    label_ids.append(self.label2id[word_labels[i]])
                else:
                    label_ids.append(-100)
        tokens    = tokens[:self.max_len - 2]
        label_ids = label_ids[:self.max_len - 2]

        # Thêm CLS / SEP
        input_ids = self.tokenizer.convert_tokens_to_ids(
            [self.tokenizer.cls_token] + tokens + [self.tokenizer.sep_token]
        )
        attention_mask = [1] * len(input_ids)
        label_ids      = [-100] + label_ids + [-100]   # CLS=−100, SEP=−100

        # Padding
        pad_len         = self.max_len - len(input_ids)
        input_ids      += [self.tokenizer.pad_token_id] * pad_len
        attention_mask += [0] * pad_len
        label_ids      += [-100] * pad_len

        return {
            'input_ids':      torch.tensor(input_ids,      dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels':         torch.tensor(label_ids,      dtype=torch.long),
        }


In [ ]:
def make_collate_fn(max_len):
    def custom_collate(batch):
        input_ids      = torch.stack([item['input_ids']      for item in batch])
        attention_mask = torch.stack([item['attention_mask'] for item in batch])
        labels         = torch.stack([item['labels']         for item in batch])
        return {
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'labels':         labels,
        }
    return custom_collate


In [ ]:
def build_window_graph(seq_lens, max_len: int, window: int = 4, device='cpu'):

    rows, cols = [], []

    for b, sl in enumerate(seq_lens):
        sl     = int(sl)
        offset = b * max_len

        for i in range(sl):
            for delta in range(-window, window + 1):
                if delta == 0:
                    continue
                j = i + delta
                if j < 0 or j >= sl:
                    continue
                rows.append(i + offset)
                cols.append(j + offset)

    if not rows:
        return torch.zeros((2, 0), dtype=torch.long, device=device)

    return torch.tensor([rows, cols], dtype=torch.long, device=device)


In [ ]:
def build_bio_constraint(label2id: dict) -> torch.Tensor:
    num_labels = len(label2id)

    # Mặc định tất cả transition đều hợp lệ
    constraint = torch.ones(num_labels, num_labels, dtype=torch.bool)

    for from_label, from_id in label2id.items():
        for to_label, to_id in label2id.items():

            # Xác định prefix và type của from
            if from_label == 'O':
                from_prefix = 'O'
                from_type   = None
            elif from_label.startswith('B-'):
                from_prefix = 'B'
                from_type   = from_label[2:]
            elif from_label.startswith('I-'):
                from_prefix = 'I'
                from_type   = from_label[2:]
            else:
                # Nhãn lạ không theo BIO → cho phép tất cả
                continue

            # Xác định prefix và type của to
            if to_label == 'O':
                to_prefix = 'O'
                to_type   = None
            elif to_label.startswith('B-'):
                to_prefix = 'B'
                to_type   = to_label[2:]
            elif to_label.startswith('I-'):
                to_prefix = 'I'
                to_type   = to_label[2:]
            else:
                continue

            # ── Áp dụng các rule ──────────────────────────────
            valid = True

            # Rule 3: O → I-* KHÔNG OK
            if from_prefix == 'O' and to_prefix == 'I':
                valid = False

            # Rule 5: B-X → I-Y (khác type) KHÔNG OK
            elif from_prefix == 'B' and to_prefix == 'I':
                if from_type != to_type:
                    valid = False

            # Rule 9: I-X → I-Y (khác type) KHÔNG OK
            elif from_prefix == 'I' and to_prefix == 'I':
                if from_type != to_type:
                    valid = False

            constraint[from_id][to_id] = valid

    return constraint   # (num_labels, num_labels)


def build_bio_start_constraint(label2id: dict) -> torch.Tensor:

    num_labels = len(label2id)
    start_constraint = torch.ones(num_labels, dtype=torch.bool)

    for label, idx in label2id.items():
        if label.startswith('I-'):
            start_constraint[idx] = False  # Không được bắt đầu câu bằng I-

    return start_constraint  # (num_labels,)


In [ ]:
class PhoBERT_GAT_CRF(nn.Module):

    def __init__(self, phobert_name, hidden_dim=256, num_labels=10,
                 gat_heads=8, dropout_prob=0.3, window_size=4,
                 use_decoder=True, label2id=None):
        super().__init__()
        self.use_decoder = use_decoder
        self.window_size = window_size

        # Encoder
        self.encoder = AutoModel.from_pretrained(phobert_name)
        self.dropout  = nn.Dropout(dropout_prob)
        self.proj     = nn.Linear(self.encoder.config.hidden_size, hidden_dim)

        # GAT
        assert hidden_dim % gat_heads == 0
        self.gat = GATv2Conv(
            hidden_dim,
            hidden_dim // gat_heads,
            heads=gat_heads,
            dropout=dropout_prob,
            concat=True
        )

        # Transformer Decoder
        if self.use_decoder:
            decoder_layer = TransformerDecoderLayer(
                d_model=hidden_dim, nhead=8,
                dropout=dropout_prob, batch_first=True
            )
            self.decoder = TransformerDecoder(decoder_layer, num_layers=1)

        # Classifier + CRF
        self.classifier = nn.Linear(hidden_dim, num_labels)
        self.crf        = CRF(num_labels, batch_first=True)

        # ── BIO Constraint ──────
        if label2id is not None:
            trans_mask = build_bio_constraint(label2id)
            start_mask = build_bio_start_constraint(label2id)
        else:
            trans_mask = torch.ones(num_labels, num_labels, dtype=torch.bool)
            start_mask = torch.ones(num_labels, dtype=torch.bool)
        self.register_buffer('bio_trans_mask',  trans_mask)
        self.register_buffer('bio_start_mask',  start_mask)

    def _apply_bio_constraints(self):

        NEG_INF = -1e4

        with torch.no_grad():
            self.crf.transitions.data[~self.bio_trans_mask] = NEG_INF

            self.crf.start_transitions.data[~self.bio_start_mask] = NEG_INF

    def forward(self, input_ids, attention_mask, labels=None):
        # ── 1. PhoBERT encoder ───────────────────────────────
        h = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state
        h = self.dropout(h)
        h = self.proj(h)                               # (B, T, hidden_dim)

        batch_size, seq_len, hidden_dim = h.size()
        h_flat = h.view(-1, hidden_dim)                # (B*T, hidden_dim)

        # ── 2. Window Graph ──────────────────────────────────
        seq_lens   = attention_mask.sum(dim=1).tolist()
        edge_index = build_window_graph(
            seq_lens, seq_len,
            window=self.window_size,
            device=h_flat.device
        )

        # ── 3. GAT ──────────────────────────────────────────
        h_gat = self.gat(h_flat, edge_index)
        h     = h_gat.view(batch_size, seq_len, hidden_dim)

        # ── 4. Transformer Decoder (optional) ────────────────
        if self.use_decoder:
            tgt_key_padding_mask = ~attention_mask.bool()
            h = self.decoder(
                tgt=h, memory=h,
                tgt_key_padding_mask=tgt_key_padding_mask
            )

        # ── 5. Classifier emissions ──────────────────────────
        emissions = self.classifier(self.dropout(h))   # (B, T, num_labels)

        # Bỏ CLS (index 0)
        emissions_no_cls = emissions[:, 1:, :]         # (B, T-1, C)

        if labels is not None:
            # ── TRAINING: không apply constraint ──────────────
            labels_no_cls = labels[:, 1:].clone()
            mask_no_cls   = (labels_no_cls != -100)
            labels_no_cls[~mask_no_cls] = 0

            loss = -self.crf(
                emissions_no_cls, labels_no_cls,
                mask=mask_no_cls, reduction='mean'
            )
            return loss

        # ── INFERENCE: apply BIO constraint trước khi decode ──
        # Lưu lại giá trị gốc để restore sau (tránh ảnh hưởng training tiếp theo)
        orig_trans = self.crf.transitions.data.clone()
        orig_start = self.crf.start_transitions.data.clone()

        self._apply_bio_constraints()

        mask_no_cls = attention_mask[:, 1:].bool()
        pred_seqs   = self.crf.decode(emissions_no_cls, mask=mask_no_cls)

        # Restore về giá trị gốc (quan trọng nếu còn training sau eval)
        self.crf.transitions.data.copy_(orig_trans)
        self.crf.start_transitions.data.copy_(orig_start)

        return pred_seqs  # List[List[int]]


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device, grad_clip=None):
    model.train()
    total_loss = 0.0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = model(input_ids, attention_mask, labels)
        loss.backward()

        if grad_clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def eval_model(model, loader, id2label, device):

    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            pred_seqs = model(input_ids, attention_mask)   # List[List[int]]

            for p_seq, t_seq in zip(pred_seqs, labels.tolist()):
                t_no_cls = t_seq[1:]

                p_ptr = 0
                for t_val in t_no_cls:
                    if t_val == -100:
                        p_ptr += 1
                        continue
                    if p_ptr >= len(p_seq):
                        break
                    trues.append(merge_entity_label(id2label[t_val]))
                    preds.append(merge_entity_label(id2label[p_seq[p_ptr]]))
                    p_ptr += 1

    labels_list = sorted({l for l in set(trues) if l != 'O'})
    precision, recall, f1, _ = precision_recall_fscore_support(
        trues, preds, average=None, labels=labels_list, zero_division=0
    )
    macro_f1 = f1_score(trues, preds, average='macro',
                        labels=labels_list, zero_division=0)
    micro_f1 = f1_score(trues, preds, average='micro',
                        labels=labels_list, zero_division=0)
    report   = classification_report(trues, preds, labels=labels_list, digits=4)
    label_scores = {
        label: {'precision': float(p), 'recall': float(r), 'f1': float(f)}
        for label, p, r, f in zip(labels_list, precision, recall, f1)
    }
    return report, label_scores, macro_f1, micro_f1


In [ ]:
train_s, train_l = load_conll_data("/content/drive/MyDrive/ViMedNer/data/train.txt")
dev_s,   dev_l   = load_conll_data("/content/drive/MyDrive/ViMedNer/data/dev.txt")
test_s,  test_l  = load_conll_data("/content/drive/MyDrive/ViMedNer/data/test.txt")

all_labels = sorted({label for seq in train_l + dev_l + test_l for label in seq})
label2id   = {l: i for i, l in enumerate(all_labels)}
id2label   = {i: l for l, i in label2id.items()}

with open(f"{OUTPUT_DIR}/label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

print(f"Labels ({len(all_labels)}): {all_labels}")

tokenizer  = AutoTokenizer.from_pretrained(PHOBERT)
collate_fn = make_collate_fn(MAX_LEN)

train_loader = DataLoader(
    TextGraphDataset(train_s, train_l, tokenizer, label2id, MAX_LEN),
    batch_size=BATCH, shuffle=True, collate_fn=collate_fn
)
dev_loader = DataLoader(
    TextGraphDataset(dev_s, dev_l, tokenizer, label2id, MAX_LEN),
    batch_size=BATCH, collate_fn=collate_fn
)
test_loader = DataLoader(
    TextGraphDataset(test_s, test_l, tokenizer, label2id, MAX_LEN),
    batch_size=BATCH, collate_fn=collate_fn
)
print(f"Train: {len(train_loader.dataset)} | Dev: {len(dev_loader.dataset)} | Test: {len(test_loader.dataset)}")



In [ ]:
model = PhoBERT_GAT_CRF(
    phobert_name=PHOBERT,
    hidden_dim=HIDDEN_DIM,
    num_labels=len(all_labels),
    gat_heads=GAT_HEADS,
    dropout_prob=DROP_PROB,
    window_size=WINDOW_SIZE,
    use_decoder=True,
    label2id=label2id
).to(device)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = EPOCHS * len(train_loader)
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

best_macro_f1   = 0.0
best_model_path = f"{OUTPUT_DIR}/best_model.pt"
history         = []

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")


In [ ]:
for ep in range(1, EPOCHS + 1):
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler, device, grad_clip=GRAD_CLIP
    )
    report, label_scores, macro_f1, micro_f1 = eval_model(
        model, dev_loader, id2label, device
    )

    print(f"\nEpoch {ep}/{EPOCHS} — Train loss: {train_loss:.4f}")
    print(report)
    print(f"Macro-F1: {macro_f1:.4f} | Micro-F1: {micro_f1:.4f}")

    history.append({
        "epoch":      ep,
        "train_loss": round(train_loss, 6),
        "macro_f1":   round(macro_f1,   6),
        "micro_f1":   round(micro_f1,   6),
    })

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save({
            "epoch":       ep,
            "model_state": model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "macro_f1":    macro_f1,
            "micro_f1":    micro_f1,
        }, best_model_path)
        print(f"  ✅ Best model saved — Macro-F1={best_macro_f1:.4f}")

with open(f"{OUTPUT_DIR}/train_history.json", "w") as f:
    json.dump(history, f, indent=2)
print("\n✅ Training done!")


In [ ]:
print("=== Test Set Evaluation (best model) ===")
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state"])
print(f"Best checkpoint: epoch {checkpoint['epoch']} | Dev Macro-F1={checkpoint['macro_f1']:.4f}")

report, label_scores, macro_f1, micro_f1 = eval_model(model, test_loader, id2label, device)
print(report)
print(f"Test Macro-F1: {macro_f1:.4f} | Test Micro-F1: {micro_f1:.4f}")

with open(f"{OUTPUT_DIR}/test_report.txt", "w", encoding="utf-8") as f:
    f.write(f"Best checkpoint : epoch {checkpoint['epoch']}\n")
    f.write(f"Dev  Macro-F1   : {checkpoint['macro_f1']:.4f}\n")
    f.write(f"Test Macro-F1   : {macro_f1:.4f} | Micro-F1: {micro_f1:.4f}\n\n")
    f.write(report)

with open(f"{OUTPUT_DIR}/test_label_scores.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["label", "precision", "recall", "f1"])
    writer.writeheader()
    for label, scores in label_scores.items():
        writer.writerow({
            "label":     label,
            "precision": round(scores["precision"], 4),
            "recall":    round(scores["recall"],    4),
            "f1":        round(scores["f1"],        4),
        })
print("✅ Saved: test_report.txt | test_label_scores.csv")


In [ ]:
model.eval()
all_preds_bio, all_trues_bio = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        pred_seqs = model(input_ids, attention_mask)       # List[List[int]]

        for p_seq, t_seq in zip(pred_seqs, labels.tolist()):
            t_no_cls = t_seq[1:]
            result_t, result_p = [], []
            p_ptr = 0
            for t_val in t_no_cls:
                if t_val == -100:
                    p_ptr += 1
                    continue
                if p_ptr >= len(p_seq):
                    break
                result_t.append(id2label[t_val])
                result_p.append(id2label[p_seq[p_ptr]])
                p_ptr += 1
            all_trues_bio.append(result_t)
            all_preds_bio.append(result_p)

pred_path = f"{OUTPUT_DIR}/test_predictions.txt"
with open(pred_path, "w", encoding="utf-8") as f:
    for s_words, s_trues, s_preds in zip(test_s, all_trues_bio, all_preds_bio):
        for word, true_tag, pred_tag in zip(s_words, s_trues, s_preds):
            f.write(f"{word}\t{true_tag}\t{pred_tag}\n")
        f.write("\n")
print(f"✅ Predictions saved → {pred_path}")


In [ ]:
def plot_confusion_matrix(trues, preds, labels, title, save_path, annot_threshold=20):
    n         = len(labels)
    label2idx = {l: i for i, l in enumerate(labels)}

    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(trues, preds):
        if t in label2idx and p in label2idx:
            cm[label2idx[t]][label2idx[p]] += 1

    cell_size = max(0.7, min(1.2, 18 / n))
    fig_size  = max(10, n * cell_size)
    annot     = n <= annot_threshold

    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))
    sns.heatmap(
        cm, annot=annot, fmt='d', cmap='magma',
        xticklabels=labels, yticklabels=labels, ax=ax,
        annot_kws={"size": max(7, 11 - n // 5)} if annot else {}
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True',      fontsize=12)
    ax.set_title(title,        fontsize=13)
    plt.xticks(rotation=45, ha='right', fontsize=max(7, 10 - n // 8))
    plt.yticks(rotation=0,              fontsize=max(7, 10 - n // 8))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved → {save_path}")


flat_trues_all = [t for sent in all_trues_bio for t in sent]
flat_preds_all = [p for sent in all_preds_bio for p in sent]

# Full matrix
labels_all = sorted(set(flat_trues_all) | set(flat_preds_all))
plot_confusion_matrix(
    trues=flat_trues_all, preds=flat_preds_all, labels=labels_all,
    title='Confusion Matrix — FULL (Including O)',
    save_path=f"{OUTPUT_DIR}/confusion_matrix_full.png"
)

# Entity-only (true != O)
pairs_e      = [(t, p) for t, p in zip(flat_trues_all, flat_preds_all) if t != 'O']
flat_trues_e = [t for t, p in pairs_e]
flat_preds_e = [p for t, p in pairs_e]
labels_e     = sorted(set(flat_trues_e) | set(flat_preds_e))
plot_confusion_matrix(
    trues=flat_trues_e, preds=flat_preds_e, labels=labels_e,
    title='Confusion Matrix — ENTITY ONLY (true != O)',
    save_path=f"{OUTPUT_DIR}/confusion_matrix_entity_only.png"
)

# Entity-strict (cả hai != O)
pairs_strict  = [(t, p) for t, p in zip(flat_trues_all, flat_preds_all)
                 if t != 'O' and p != 'O']
flat_trues_st = [t for t, p in pairs_strict]
flat_preds_st = [p for t, p in pairs_strict]
labels_st     = sorted(set(flat_trues_st) | set(flat_preds_st))
plot_confusion_matrix(
    trues=flat_trues_st, preds=flat_preds_st, labels=labels_st,
    title='Confusion Matrix — ENTITY STRICT (both != O)',
    save_path=f"{OUTPUT_DIR}/confusion_matrix_entity_strict.png"
)


In [ ]:
# ── Vẽ biểu đồ Training History ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

epochs_x   = [h["epoch"]      for h in history]
losses     = [h["train_loss"] for h in history]
macro_f1s  = [h["macro_f1"]   for h in history]
micro_f1s  = [h["micro_f1"]   for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History", fontsize=14, fontweight='bold', y=1.02)

# ── Plot 1: Train Loss ───────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(epochs_x, losses, color='#E24B4A', linewidth=2,
         marker='o', markersize=5, label='Train Loss')

# Đánh dấu điểm loss thấp nhất
min_loss_ep = epochs_x[losses.index(min(losses))]
min_loss_v  = min(losses)
ax1.axvline(min_loss_ep, color='#E24B4A', linestyle='--', alpha=0.4)
ax1.annotate(
    f"min={min_loss_v:.4f}\nep={min_loss_ep}",
    xy=(min_loss_ep, min_loss_v),
    xytext=(min_loss_ep + 0.4, min_loss_v + (max(losses) - min_loss_v) * 0.15),
    fontsize=9, color='#A32D2D',
    arrowprops=dict(arrowstyle='->', color='#A32D2D', lw=1.2)
)

ax1.set_title("Train Loss per Epoch", fontsize=12)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.legend(fontsize=10)

# ── Plot 2: F1 Score ─────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(epochs_x, macro_f1s, color='#185FA5', linewidth=2,
         marker='o', markersize=5, label='Macro F1')
ax2.plot(epochs_x, micro_f1s, color='#1D9E75', linewidth=2,
         marker='s', markersize=5, linestyle='--', label='Micro F1')

# Đánh dấu best Macro F1
best_ep  = epochs_x[macro_f1s.index(max(macro_f1s))]
best_val = max(macro_f1s)
ax2.axvline(best_ep, color='#185FA5', linestyle='--', alpha=0.4)
ax2.annotate(
    f"best={best_val:.4f}\nep={best_ep}",
    xy=(best_ep, best_val),
    xytext=(best_ep + 0.4, best_val - (max(macro_f1s) - min(macro_f1s)) * 0.2),
    fontsize=9, color='#0C447C',
    arrowprops=dict(arrowstyle='->', color='#0C447C', lw=1.2)
)

ax2.set_title("F1 Score per Epoch (Dev Set)", fontsize=12)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1 Score")
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax2.set_ylim(0, 1.05)
ax2.grid(True, linestyle='--', alpha=0.4)
ax2.legend(fontsize=10)

plt.tight_layout()
plot_path = f"{OUTPUT_DIR}/training_history.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → {plot_path}")